# Datamine AUTOCR Process Example

This notebook demonstrates how to configure and run the **`autocr`** process wrapper in `dmstudio`.

## Process Description

## AUTOCR

Auto-correlation analysis is used to quantify and define anomalous thresholds and halo size on regularly grided soil sample lines. **AUTOCR** can also be used to measure dispersion limits in stream sediments, as long as fixed distances are used for sampling.

The process calculates the auto correlation function **R-L** of a single field against sample **DISTANCE** or **LAG** (see Figure 1). By default, lag distance is used to calculate the auto correlation function. If sample distance, or a multiple of sample distance is required, use the parameter **SAMPDIST**.

Anomalous samples related to lag or sample distance (**LAG** or **DISTANCE**) are identified by strong and well defined peaks of **R-L** , the covariance between neighboring samples. Dispersion limits from mineralized stream sediments are defined by breaks in slope along the stream sediment train as defined by **LAG-L**.

**Note** : There is a restriction of 4000 samples for a given line or dispersion train. For the analysis to be valid the data points must be equi-distant from each other. All fields containing the same value should be removed prior to the analysis. A plot macro should be written to display the results.

### Input Files:

* **in** (Undefined):
  Input file. Must contain sample identity field.
  Required=Yes

### Output Files:

* **out** (Undefined):
  Output file includes LAG-L, DISTANCE. R-L the auto correlation function and SIGNIC the
  significance of the auto correlation function for use in graphical processes.
  Required=Yes

### Fields:

* **sampid** (Undefined : IN):
  Compulsory sample identifier field.
  Default=Undefined
  Required=Yes

* **fields** (Undefined : Undefined):
  First variable for evaluation. If no variables are selected all variables will be processed.
  Default=Undefined
  Required=No

### Parameters:

* **sampdist**:

* **Options** ((0): Distance between sample points to calculate the auto-correlation):
  function. If no distance is specified the sample distance is lag distance.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **print**:
  >0 Display results on the screen (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('autocr')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute autocr
print("Running autocr...")
dm_cmd.autocr(
    in_i='t_assays',  # required
    out_o='t_autocr_out',  # required
    sampid_f='optional',  # required
    fields_f=['AU'],  # required
    # sampdist_p=0,  # optional
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("autocr execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_autocr_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")